In [1]:
bronze_df = spark.table("bronze_stock_prices")

display(bronze_df)

StatementMeta(, 48240ff8-833e-487d-8cb1-2eba15f05899, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 781eace5-6ec0-4c59-ba70-9ebd4e18251e)

In [2]:
print(
    f"Bronze row count: {bronze_df.count()}"
)

StatementMeta(, 48240ff8-833e-487d-8cb1-2eba15f05899, 4, Finished, Available, Finished, False)

Bronze row count: 1103


In [3]:
# Chck the schema
bronze_df.printSchema()

StatementMeta(, 48240ff8-833e-487d-8cb1-2eba15f05899, 5, Finished, Available, Finished, False)

root
 |-- Date: timestamp (nullable = true)
 |-- Close: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- ticker: string (nullable = true)



In [4]:
# Check for duplicates by date and ticket
from pyspark.sql import functions as F

duplicate_count = (
    bronze_df
    .groupBy("Date", "ticker")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate combinations: {duplicate_count}")

StatementMeta(, 48240ff8-833e-487d-8cb1-2eba15f05899, 6, Finished, Available, Finished, False)

Duplicate combinations: 0


In [5]:
# Check for Nulls

bronze_df.select(
    [
        F.count(
            F.when(
                F.col(c).isNull(),
                c
            )
        ).alias(c)
        for c in bronze_df.columns
    ]
).show()

StatementMeta(, 48240ff8-833e-487d-8cb1-2eba15f05899, 7, Finished, Available, Finished, False)

+----+-----+----+---+----+------+------+
|Date|Close|High|Low|Open|Volume|ticker|
+----+-----+----+---+----+------+------+
|   0|    0|   0|  0|   0|     0|     0|
+----+-----+----+---+----+------+------+



In [6]:
# Drop any duplicates

silver_df = bronze_df.dropDuplicates(["Date", "ticker"])

StatementMeta(, 48240ff8-833e-487d-8cb1-2eba15f05899, 8, Finished, Available, Finished, False)

In [7]:
# Drop rows with critical nulls

silver_df = silver_df.dropna(
    subset=[
        "Date",
        "ticker",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
)

StatementMeta(, 48240ff8-833e-487d-8cb1-2eba15f05899, 9, Finished, Available, Finished, False)

In [8]:
# Verify data cleaning

print(
    f"Bronze Count: {bronze_df.count()}"
)

print(
    f"Silver Count: {silver_df.count()}"
)

StatementMeta(, 48240ff8-833e-487d-8cb1-2eba15f05899, 10, Finished, Available, Finished, False)

Bronze Count: 1103
Silver Count: 1103


In [9]:
# Data Quality Validation

silver_df.select(
    [
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in silver_df.columns
    ]
).show()

StatementMeta(, 48240ff8-833e-487d-8cb1-2eba15f05899, 11, Finished, Available, Finished, False)

+----+-----+----+---+----+------+------+
|Date|Close|High|Low|Open|Volume|ticker|
+----+-----+----+---+----+------+------+
|   0|    0|   0|  0|   0|     0|     0|
+----+-----+----+---+----+------+------+



In [13]:
# Write to Sillver Delta table in the Lakehouse
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_stock_prices")

StatementMeta(, 48240ff8-833e-487d-8cb1-2eba15f05899, 15, Finished, Available, Finished, False)

In [14]:
# Validate Silver Delta table load by displaying a sample of cleaned records

display(
    spark.sql("""
        SELECT *
        FROM silver_stock_prices
        LIMIT 10
    """)
)

StatementMeta(, 48240ff8-833e-487d-8cb1-2eba15f05899, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 272b54c6-6f26-4c79-9332-ec35d66c3c31)